<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> 一书的补充代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第 3 章：练习题解答

本 notebook 中使用的包：

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.4
torch version: 2.7.1
tokenizers version: 0.21.4


&nbsp;
## 练习 3.1：添加更多测试用例

- 我们可以添加无穷多的不同测试用例
- 下面是一些有趣的选择

In [ ]:
from reasoning_from_scratch.ch03 import (
    run_demos_table
)

more_tests = [
    # 不同的括号类型
    ("check_17", "[1, 2]", "(1, 2)", True),

    # 科学计数法
    ("check_18", "1e-3", "0.001", True),

    # 使用脱字符指数的代数化简
    ("check_19", "(-3)^2", "9", True),

    # Unicode 减号 (U+2212) vs ASCII hyphen-minus
    ("check_20", "−1", "-1", True),

]

run_demos_table(more_tests)

Test     | Expect | Got   | Status
check_17 | True   | True  | PASS  
check_18 | True   | True  | PASS  
check_19 | True   | True  | PASS  
check_20 | True   | False | FAIL  

Passed 3/4


- 如我们所见，除了 `check_20` 之外，所有测试用例都通过了，该测试将常规符号与一个看起来与人眼无法区分的 Unicode 版本的减号进行了交换
- 我们可以通过在 `normalize_text` 函数的任意位置添加以下行之一来修复此测试用例

```python
text = text.replace("−", "-")
# 或者
text = text.replace("\u2212", "-")
```

- 乍一看，另一个有趣的测试是：

In [2]:
extra_tests_1 = [
    ("check_21", "Text around answer 3.", "3", True)
]

run_demos_table(extra_tests_1)

Test     | Expect | Got   | Status
check_21 | True   | False | FAIL  

Passed 0/1


- 虽然看起来我们的代码无法处理这种包含文本的情况，但这实际上是一个设计不当的测试
- 在实践中，`run_demos_table` 函数专门用于测试 `grade_answer` 函数；仅此而已
- `grade_answer` 函数永远不会以这种形式接收完整答案，因为答案在传递给它之前就已经从文本中提取出来了

即，如果我们想测试文本答案，我们需要按如下方式调用测试：

In [3]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate
)


extra_tests_2 = [
    ("check_21",
     extract_final_candidate("Text around answer 3."),
     "3", True)
]
run_demos_table(extra_tests_2)

Test     | Expect | Got  | Status
check_21 | True   | True | PASS  

Passed 1/1


&nbsp;
## 练习 3.2：计算平均响应长度

- 选项 A：我们可以通过添加以下行来修改 `evaluate_math500_stream` 函数：

```python
# ...
# 在 `num_correct = 0` 下方
total_len = 0

# ...
# 在 for i, row in enumerate(math_data, start=1): 内部
# `gen_text = ...` 下方的任意位置
total_len += len(tokenizer.encode(gen_text))

# ...
# 在 return 语句之前底部的任意位置
avg_len = total_len / num_examples
print(f"Average length: {avg_len:.2f} tokens")
```

- 或者，我们也可以从主章节中运行 `evaluate_math500_stream` 函数时创建的 `.jsonl` 文件计算响应长度
- 首先，按如下方式加载 `.jsonl` 文件：

In [5]:
import json
from pathlib import Path

WHICH_MODEL = "base"

dev_name = "mps"  # e.g., "cuda", "cpu"

# 你可能需要调整此路径：
local_path = Path(f"math500-{dev_name}.jsonl")
if not local_path.exists():
    raise FileNotFoundError(
        f"{local_path} not found. Run ch03_main.ipynb to create it."
    )

results = []
with open(local_path, "r") as f:
    for line in f:
        if line.strip():
            results.append(json.loads(line))

print("Number of entries:", len(results))


Number of entries: 10


- 注意每个条目有多个键，但是我们只关心 `"generated_text"` 键，它包含模型的完整答案：

In [6]:
print(results[0].keys())

dict_keys(['index', 'problem', 'gtruth_answer', 'generated_text', 'extracted', 'correct'])


- 注意每个条目有多个键；但是我们只关心 `"generated_text"` 键，它包含模型的完整答案：

In [7]:
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer
)

if WHICH_MODEL == "base":

    download_qwen3_small(
        kind="base", tokenizer_only=True, out_dir="qwen3"
    )
    tokenizer_path = Path("qwen3") / "tokenizer-base.json"
    tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

elif WHICH_MODEL == "reasoning":

    download_qwen3_small(
        kind="reasoning", tokenizer_only=True, out_dir="qwen3"
    )
    tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_path,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )

✓ qwen3/tokenizer-base.json already up-to-date


- 然后，我们可以按如下方式计算平均长度，这类似于我们修改 `evaluate_math500_stream` 函数的方式：

In [8]:
total_len = 0

for item in results:
    num_tokens = len(tokenizer.encode(item["generated_text"]))
    total_len += num_tokens

avg_len = total_len / len(results)
print(f"Average length: {avg_len:.2f} tokens")

Average length: 98.00 tokens


| 模式      | 设备  | 平均长度 | MATH-500 大小  |
|-----------|---------|----------------|----------------|
| 基座      | CPU     | 97.3           | 10             |
| 基座      | MPS     | 98.0           | 10             |
| 推理 | CPU     | 891.80         | 10             |
| 推理 | MPS     | 1159.30        | 10             |
|           |         |                |                |
| 基座      | CUDA    | 96.74          | 500            |
| 推理 | CUDA    | 1361.21        | 500            |

- 正如我们所见，且正如预期的那样，推理模型的响应要长得多

&nbsp;
## 练习 3.3：扩展或更改评估数据集

- 要在更大的数据集上评估模型，我们只需将 `math_data[:10]` 更改为不同的切片或更大的数字（最多 500）

```python
num_correct, num_examples, acc = evaluate_math500_stream(
    model, tokenizer, device, 
    math_data=math_data[:10],
    max_new_tokens=2048,
    verbose=False
)
```

- 下表显示了不同数据集大小的准确率值（由于 MATH-500 测试集已经打乱，未进行额外打乱）

| 模式      | 设备  | 准确率 | MATH-500 大小  |
|-----------|---------|----------|----------------|
| 基座      | CUDA    | 30.0%    | 10             |
| 基座      | CUDA    | 34.0%    | 50             |
| 基座      | CUDA    | 27.0%    | 100            |
| 基座      | CUDA    | 31.0%    | 200            |
| 基座      | CUDA    | 15.3%    | 500            |
|           |         |          |                |
| 推理 | CUDA    | 90.0%    | 10             |
| 推理 | CUDA    | 58.0%    | 50             |
| 推理 | CUDA    | 58.0%    | 100            |
| 推理 | CUDA    | 56.0%    | 200            |
| 推理 | CUDA    | 48.2%    | 500            |

- 从上面的结果可以看出，前 10 个示例并不能很好地代表在全部 500 个示例上评估的 MATH-500 性能

- 此外，我们可以创建一个与 MATH-500 风格类似的全新数据集
- 例如，本仓库中包含一个 MATH-500 风格的数据集；我们可以在主章节中通过将文件名从 `math500_test.json` 更改为 `math_new50_exercise.json` 来使用它（该数据集包含在本书的 GitHub 仓库中：https://github.com/rasbt/reasoning-from-scratch/tree/main/ch03/01_main-chapter-code）
- 基座模型和推理模型的性能如下：
    - 基座：36.0%（18/50）
    - 推理：80.0%（40/50）
- 由此我们可以得出结论，虽然原始 MATH-500 测试数据集可能已被包含在 Qwen3 的训练数据中，但该模型在新的数学问题上表现类似，这表明它没有对原始 MATH-500 数据过度过拟合（overfitting）

&nbsp;
## 练习 3.4：尝试不同的提示模板

- 我们可以使用类似于章节中建议的替代提示，将提示修改为使用 "Problem" 而非 "Question"：

```python
def render_prompt(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "Solve the problem and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"
        f"Problem:\n{prompt}\n\nAnswer:"
    )
    return template
```

- 使用此提示可以将基座模型在 500 个示例上的性能从 15.3% 提升至 31.2%
- 反之，它会将推理模型的性能从 50.8% 降低至 50.0%
- 从这些观察中，我们可以得出结论：基座模型对提示格式更加敏感（可能是因为它从训练集中记住了一些使用提示格式的 MATH-500 示例），而推理模型则基本不受影响